# 08 — Context Compression：对话历史压缩

随着 Agent 运行，对话历史会持续增长。这章实现**自动压缩机制**：在接近 token 上限时，把历史摘要成几百 token，让 Agent 能继续运行而不报错。

## 1. 为什么需要压缩

gpt-oss:120b 的上下文窗口约 128k token。一个 Agent 任务很容易就撑满：

- 读了 10 个文件（每个 500 token）= 5,000 token
- 20 轮工具调用（每轮 2,000 token）= 40,000 token
- 系统 prompt + 对话本身 = 轻松超 100k

超限直接报错，所以要在达到 80% 时主动压缩。

```
压缩前（约 100k tokens）：
┌────────────────────────────────────────────────────────────┐
│ [system]                                         ~500 tok  │
│ [user1] 帮我重构这个项目                          ~50 tok   │
│ [ai1] 好的，我先看看目录结构                       ~80 tok   │
│ [tool: list_directory] → 结果                    ~200 tok  │
│ [tool: read_file src/main.py] → 文件内容          ~800 tok  │
│ [tool: read_file src/utils.py] → 文件内容         ~600 tok  │
│ [ai2] 我发现几个问题... tool_call: write_file     ~500 tok  │
│ ... 继续20轮 ...                                  ~90k tok  │
└────────────────────────────────────────────────────────────┘

压缩后（约 3k tokens）：
┌─────────────────────────────────────────────────┐
│ [system]                             ~500 tok    │
│ [assistant] [对话历史摘要]                        │
│   任务：重构项目。已读取 main.py、utils.py。       │
│   已发现：函数 parse_data 有内存泄漏。            │
│   已完成：重写 src/main.py，修复了3处 bug。       │
│   待处理：tests/ 目录下的单元测试还未更新。        │
│                                      ~2k tok     │
│ [user] 继续（当前轮次）               ~20 tok     │
└─────────────────────────────────────────────────┘
```

压缩后 LLM 仍然知道：做了什么、发现了什么、改了哪些文件、接下来要干什么。

In [ ]:
import sys
sys.path.insert(0, "..")

from src.llm_client import LLMClient
from src.context_manager import ContextManager, MessageItem, UsageStats, count_tokens, build_system_prompt

print("导入成功")
print(f"count_tokens('hello world') = {count_tokens('hello world')}")

## 2. format_history_for_compression

压缩的第一步：把 `MessageItem` 列表格式化为纯文本，供 LLM 阅读和摘要。

关键处理：
- 跳过 `system` 消息（系统 prompt 不压缩，保持原样）
- `tool` 消息内容通常很大（文件内容等），截断到 500 字符
- 格式清晰：`[角色]: 内容`，LLM 能快速理解对话结构

In [ ]:
def format_history_for_compression(messages: list) -> str:
    """
    把历史消息列表格式化为可供压缩的纯文本。
    
    - 跳过 system 消息
    - user/assistant 消息：保留原文
    - tool 消息：截断到 500 字符（工具结果通常很大）
    """
    MAX_TOOL_CHARS = 500  # 工具结果截断长度
    lines = []
    
    for msg in messages:
        role = msg.role
        content = msg.content
        
        # 系统消息不参与压缩
        if role == "system":
            continue
        
        # 工具结果截断
        if role == "tool":
            if len(content) > MAX_TOOL_CHARS:
                content = content[:MAX_TOOL_CHARS] + f"\n...[工具结果已截断，原长 {len(msg.content)} 字符]"
        
        # assistant 有工具调用时，补充工具调用信息
        if role == "assistant" and msg.tool_calls:
            tool_names = [tc.get("function", {}).get("name", "?") for tc in msg.tool_calls]
            tool_info = f" [调用工具: {', '.join(tool_names)}]"
            content = (content or "") + tool_info
        
        lines.append(f"[{role}]: {content}")
    
    return "\n\n".join(lines)


# 构造一段模拟历史来测试
test_messages = [
    MessageItem(role="system", content="你是助手", token_count=10),
    MessageItem(role="user", content="帮我查看 src/main.py", token_count=20),
    MessageItem(
        role="assistant",
        content="我来读取这个文件",
        token_count=15,
        tool_calls=[{"function": {"name": "read_file"}}]
    ),
    MessageItem(
        role="tool",
        content="def main():\n    # 这是一个很长的文件内容 " + "x" * 600,
        token_count=200,
        tool_call_id="call_001"
    ),
    MessageItem(role="assistant", content="文件读取成功，main 函数有内存泄漏问题", token_count=25),
]

formatted = format_history_for_compression(test_messages)
print("格式化结果：")
print("-" * 60)
print(formatted)
print("-" * 60)
print(f"\n格式化后长度：{len(formatted)} 字符，{count_tokens(formatted)} tokens")

## 3. ChatCompactor

`ChatCompactor` 负责调用 LLM 生成摘要。

注意：压缩调用本身是**非流式**的——我们需要等完整摘要生成完才能替换历史，不能边生成边用。

In [ ]:
class ChatCompactor:
    """
    调用 LLM 把对话历史压缩成摘要。
    
    压缩原则：保留所有技术细节（文件名、变量名、错误信息、已完成的操作），
    删除冗余的来回确认和重复内容。
    """
    
    COMPRESSOR_SYSTEM = (
        "你是对话历史压缩器。"
        "把下面的对话历史压缩成简洁的摘要，保留所有重要的技术细节、"
        "文件名、变量名、错误信息、已执行的操作和结果。"
        "摘要要足够详细，让后续的 AI 能继续完成任务。"
        "直接输出摘要内容，不要加任何前缀或说明。"
    )
    
    def __init__(self, llm: LLMClient):
        self.llm = llm
    
    async def compress(self, ctx: ContextManager) -> tuple[str, UsageStats]:
        """
        压缩 ctx 中的历史消息，返回 (摘要字符串, token 使用量)。
        
        步骤：
        1. 格式化历史为文本
        2. 调用 LLM 生成摘要（非流式）
        3. 返回摘要和消耗的 token 数
        """
        # 1. 格式化历史
        history_text = format_history_for_compression(ctx._messages)
        
        if not history_text.strip():
            return "（无历史记录）", UsageStats()
        
        # 2. 构造压缩请求
        compression_messages = [
            {"role": "system", "content": self.COMPRESSOR_SYSTEM},
            {"role": "user", "content": history_text},
        ]
        
        # 3. 调用 LLM（非流式，等待完整摘要）
        summary, token_usage = await self.llm.chat_completion(
            compression_messages,
            stream=False,
        )
        
        usage_stats = UsageStats(
            prompt_tokens=token_usage.prompt_tokens,
            completion_tokens=token_usage.completion_tokens,
            total_tokens=token_usage.total_tokens,
        )
        
        return summary, usage_stats


print("ChatCompactor 定义完成")
print(f"压缩系统 prompt 长度: {count_tokens(ChatCompactor.COMPRESSOR_SYSTEM)} tokens")

## 4. prune_tool_outputs：压缩前先剪枝

工具输出（文件内容、搜索结果）通常是 context 中最大的块。在调用 LLM 压缩之前，先剪掉最旧的工具结果，能显著降低压缩请求本身的 token 消耗。

策略：
- 从后往前，保留最近 `protect_last_tokens` token 内的内容不动
- 对更早的消息，如果是 `role=tool` 且内容足够大，就删除（节省 `>= min_save_tokens`）
- 非工具消息（user/assistant）保留，它们是对话骨架

In [ ]:
def prune_tool_outputs(
    messages: list,
    protect_last_tokens: int = 40_000,
    min_save_tokens: int = 20_000,
) -> list:
    """
    剪掉历史中较旧的大型工具结果，减少压缩请求的 token 消耗。
    
    参数：
    - protect_last_tokens: 从最新消息往前，保留这么多 token 的内容不动
    - min_save_tokens: 删除一条工具消息至少要节省这么多 token，否则不值得删
    
    返回：剪枝后的消息列表（不修改原列表）
    """
    if not messages:
        return messages
    
    # 从后往前计算累计 token，找到「保护区」的起点
    cumulative = 0
    protect_from_idx = len(messages)  # 默认全部保护
    
    for i in range(len(messages) - 1, -1, -1):
        cumulative += messages[i].token_count
        if cumulative >= protect_last_tokens:
            protect_from_idx = i
            break
    
    # 对保护区之前的消息，删除足够大的工具结果
    result = []
    pruned_count = 0
    pruned_tokens = 0
    
    for i, msg in enumerate(messages):
        if i >= protect_from_idx:
            # 保护区内：原样保留
            result.append(msg)
        elif msg.role == "tool" and msg.token_count >= min_save_tokens:
            # 旧的大型工具结果：删除
            pruned_count += 1
            pruned_tokens += msg.token_count
        else:
            # 旧的非工具消息或小型工具结果：保留
            result.append(msg)
    
    if pruned_count > 0:
        print(f"  [剪枝] 删除了 {pruned_count} 条工具结果，节省约 {pruned_tokens} tokens")
    
    return result


# 测试剪枝效果
def make_test_message(role, content, token_count=None, tool_call_id=None):
    """快速构造测试消息"""
    tc = token_count if token_count is not None else count_tokens(content)
    return MessageItem(
        role=role,
        content=content,
        token_count=tc,
        tool_call_id=tool_call_id if role == "tool" else None,
    )

# 构造一个有大量工具结果的历史
test_history = [
    make_test_message("user", "读取所有文件"),
    make_test_message("assistant", "好的，开始读取"),
    make_test_message("tool", "文件内容A " * 5000, token_count=25_000, tool_call_id="c1"),  # 大型旧结果
    make_test_message("tool", "文件内容B " * 5000, token_count=25_000, tool_call_id="c2"),  # 大型旧结果
    make_test_message("assistant", "已读取文件 A 和 B，发现了一些问题"),
    make_test_message("user", "继续"),
    make_test_message("tool", "最新的工具结果 " * 1000, token_count=10_000, tool_call_id="c3"),  # 较新结果
    make_test_message("assistant", "分析完成"),
]

total_before = sum(m.token_count for m in test_history)
print(f"剪枝前：{len(test_history)} 条消息，{total_before} tokens")

pruned = prune_tool_outputs(test_history, protect_last_tokens=40_000, min_save_tokens=20_000)

total_after = sum(m.token_count for m in pruned)
print(f"剪枝后：{len(pruned)} 条消息，{total_after} tokens")
print(f"节省：{total_before - total_after} tokens ({(total_before - total_after) / total_before * 100:.1f}%)")

## 5. compress_if_needed：压缩触发与替换

这是对外暴露的主入口。逻辑：

1. 检查是否需要压缩（未超 80% → 直接返回 False）
2. 先剪枝（删大块旧工具结果）
3. 再调用 LLM 生成摘要
4. 清空历史，插入一条摘要消息
5. 返回 True（表示发生了压缩）

In [ ]:
async def compress_if_needed(
    ctx: ContextManager,
    compactor: ChatCompactor,
) -> bool:
    """
    检查是否需要压缩，需要则执行。
    
    返回 True 表示执行了压缩，False 表示不需要压缩。
    """
    if not ctx.needs_compression():
        return False
    
    tokens_before = ctx.estimated_tokens
    print(f"  [压缩触发] 当前 {tokens_before} tokens，超过阈值，开始压缩...")
    
    # 第一步：剪枝（删旧的大型工具结果）
    ctx._messages = prune_tool_outputs(ctx._messages)
    tokens_after_prune = ctx.estimated_tokens
    print(f"  [剪枝后] {tokens_after_prune} tokens")
    
    # 第二步：调用 LLM 生成摘要
    summary, usage = await compactor.compress(ctx)
    
    # 第三步：用摘要替换历史
    ctx.clear_messages()
    ctx.add_assistant_message(f"[对话历史摘要]\n{summary}")
    
    # 更新 token 统计（压缩本身也消耗了 token）
    ctx.update_usage(usage.prompt_tokens, usage.completion_tokens)
    
    tokens_final = ctx.estimated_tokens
    print(f"  [压缩完成] {tokens_before} → {tokens_final} tokens")
    print(f"  [压缩消耗] prompt={usage.prompt_tokens}, completion={usage.completion_tokens}")
    
    return True


print("compress_if_needed 定义完成")

## 6. 完整测试：构造长对话，触发压缩

构造一个接近 token 上限的对话，验证整个压缩流程。

In [ ]:
# 构造 100 条模拟消息（模拟大量工具调用后的 context）
def build_long_conversation(ctx: ContextManager, num_rounds: int = 20):
    """
    构造模拟对话：每轮包含用户消息、AI 消息、工具调用结果。
    工具结果模拟读取文件，内容较大。
    """
    # 模拟读取的文件内容（每份约 2000 token）
    fake_file_content = """
def process_data(df, config):
    # 数据预处理流水线
    # 步骤 1: 清洗缺失值
    df = df.dropna(subset=config['required_cols'])
    df['value'] = df['value'].fillna(df['value'].median())
    
    # 步骤 2: 特征工程
    df['log_value'] = np.log1p(df['value'])
    df['value_zscore'] = (df['value'] - df['value'].mean()) / df['value'].std()
    
    # 步骤 3: 过滤异常值（3 sigma 规则）
    mask = np.abs(df['value_zscore']) < 3
    df = df[mask]
    
    # 步骤 4: 类型转换
    for col in config['categorical_cols']:
        df[col] = df[col].astype('category')
    
    return df


def train_model(X_train, y_train, params):
    # 训练 XGBoost 模型
    model = XGBClassifier(
        n_estimators=params.get('n_estimators', 100),
        max_depth=params.get('max_depth', 6),
        learning_rate=params.get('learning_rate', 0.1),
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
    )
    model.fit(X_train, y_train, eval_set=[(X_train, y_train)], verbose=False)
    return model
""" * 3  # 重复3次让内容更大
    
    for i in range(num_rounds):
        # 用户提问
        ctx.add_user_message(f"第 {i+1} 轮：请检查 src/module_{i:02d}.py 的实现")
        
        # AI 决定调用工具
        ctx.add_assistant_message(
            f"我来读取 module_{i:02d}.py",
            tool_calls=[{"id": f"call_{i:03d}", "type": "function",
                         "function": {"name": "read_file", "arguments": f'{{"path": "src/module_{i:02d}.py"}}'}}]
        )
        
        # 工具返回（模拟文件内容）
        ctx.add_tool_result(
            tool_call_id=f"call_{i:03d}",
            content=f"# module_{i:02d}.py\n" + fake_file_content,
        )
        
        # AI 给出分析
        ctx.add_assistant_message(
            f"module_{i:02d}.py 读取完成。\n"
            f"发现：process_data 函数在第 {i*3+15} 行有潜在的内存泄漏，"
            f"train_model 的超参数可以优化。建议修改 learning_rate 从 0.1 改为 0.05。"
        )


# 初始化 context
ctx = ContextManager(system_prompt=build_system_prompt("."))
build_long_conversation(ctx, num_rounds=20)

stats = ctx.stats()
print("构造完成：")
print(f"  消息数量: {stats['message_count']} 条")
print(f"  估算 token: {stats['estimated_tokens']:,}")
print(f"  需要压缩: {ctx.needs_compression()}")
print(f"  (阈值: 128k × 0.8 = {128_000 * 0.8:,.0f} tokens)")

In [ ]:
# 展示压缩前的消息结构（只看前几条和后几条）
msgs = ctx._messages
print(f"压缩前消息列表（共 {len(msgs)} 条）：")
print()

for i, msg in enumerate(msgs[:6]):
    preview = msg.content[:60].replace("\n", " ")
    print(f"  [{i}] role={msg.role:12s} tokens={msg.token_count:5d} | {preview}...")

print(f"  ... （中间 {len(msgs) - 12} 条省略）...")

for i, msg in enumerate(msgs[-6:]):
    idx = len(msgs) - 6 + i
    preview = msg.content[:60].replace("\n", " ")
    print(f"  [{idx}] role={msg.role:12s} tokens={msg.token_count:5d} | {preview}...")

In [ ]:
# 执行压缩
llm = LLMClient()
compactor = ChatCompactor(llm)

print("开始压缩流程：")
print(f"压缩前：{ctx.estimated_tokens:,} tokens，{len(ctx._messages)} 条消息")
print()

compressed = await compress_if_needed(ctx, compactor)

print()
print(f"发生了压缩：{compressed}")
print(f"压缩后：{ctx.estimated_tokens:,} tokens，{len(ctx._messages)} 条消息")

In [ ]:
# 查看摘要内容
if ctx._messages:
    summary_msg = ctx._messages[0]
    print(f"摘要消息 role: {summary_msg.role}")
    print(f"摘要 token 数: {summary_msg.token_count}")
    print()
    print("摘要内容：")
    print("-" * 60)
    print(summary_msg.content)
    print("-" * 60)

## 7. 验证：压缩后 LLM 仍能理解上下文

压缩的核心价值在于：后续对话中 LLM 能基于摘要继续工作，而不是「失忆」。

这里验证：在压缩后的 context 里追加一条用户消息，LLM 能正确引用摘要中的信息。

In [ ]:
# 在压缩后的 context 上继续对话
ctx.add_user_message(
    "根据你之前的分析，总结一下：有哪些模块需要优化 learning_rate？"
    "以及 process_data 的内存泄漏大概在哪些行？"
)

print(f"继续对话前 context 大小: {ctx.estimated_tokens} tokens")
print()

# 发送给 LLM
reply, usage = await llm.chat_completion(ctx.get_messages(), stream=False)

ctx.add_assistant_message(reply)
ctx.update_usage(usage.prompt_tokens, usage.completion_tokens)

print("LLM 的回复：")
print("-" * 60)
print(reply)
print("-" * 60)

## 8. 错误处理：边界情况测试

In [ ]:
# 错误场景 1：空历史压缩
ctx_empty = ContextManager(system_prompt="你是助手")
print("场景 1：空历史")
result = await compress_if_needed(ctx_empty, compactor)
print(f"  needs_compression: {ctx_empty.needs_compression()}")
print(f"  压缩发生: {result}")
print()

# 错误场景 2：只有 system 消息（历史全是 system 消息）
ctx_only_system = ContextManager(system_prompt="你是助手")
ctx_only_system.add_user_message("你好")
print("场景 2：消息少，不触发压缩")
result = await compress_if_needed(ctx_only_system, compactor)
print(f"  estimated_tokens: {ctx_only_system.estimated_tokens}")
print(f"  needs_compression: {ctx_only_system.needs_compression()}")
print(f"  压缩发生: {result}")
print()

# 错误场景 3：format_history 处理空消息列表
print("场景 3：format_history 空输入")
empty_formatted = format_history_for_compression([])
print(f"  空列表结果: '{empty_formatted}'")
print()

# 错误场景 4：prune 不删除保护区内的消息
print("场景 4：prune 保护最近的消息")
recent_messages = [
    make_test_message("tool", "最新文件内容 " * 1000, token_count=30_000, tool_call_id="c1"),
]
pruned = prune_tool_outputs(recent_messages, protect_last_tokens=50_000, min_save_tokens=20_000)
print(f"  输入 {len(recent_messages)} 条 → 输出 {len(pruned)} 条（保护区内不删）")

## 9. 集成验证：模拟完整 Agent 运行

把压缩逻辑集成进一个简化的 Agent 循环，验证多次压缩能正常工作。

In [ ]:
async def simulate_agent_with_compression(
    initial_task: str,
    num_rounds: int = 5,
    compression_threshold: float = 0.001,  # 设一个极低的阈值方便触发
):
    """
    模拟一个带压缩功能的 Agent 循环。
    设置极低的 compression_threshold 来强制触发压缩，演示功能。
    """
    # 用极低阈值的 ContextManager 子类来演示
    class LowThresholdCtx(ContextManager):
        def needs_compression(self, context_window=128_000, threshold=0.001):
            # 极低阈值：只要超过 128 token 就触发
            return self.estimated_tokens > 128
    
    ctx = LowThresholdCtx(system_prompt=build_system_prompt("."))
    compactor = ChatCompactor(llm)
    
    print(f"任务：{initial_task}")
    print(f"压缩阈值：{compression_threshold * 100}% ({int(128_000 * compression_threshold)} tokens)")
    print()
    
    # 模拟 Agent 运行几轮
    for i in range(num_rounds):
        # 模拟用户/系统输入
        user_msg = f"第 {i+1} 步：请执行子任务 {i+1}" if i > 0 else initial_task
        ctx.add_user_message(user_msg)
        
        tokens_before = ctx.estimated_tokens
        
        # 检查并压缩
        compressed = await compress_if_needed(ctx, compactor)
        
        # 获取 AI 回复
        reply, usage = await llm.chat_completion(ctx.get_messages(), stream=False)
        ctx.add_assistant_message(reply[:200])  # 截断回复避免 context 过长
        ctx.update_usage(usage.prompt_tokens, usage.completion_tokens)
        
        print(f"第 {i+1} 轮：{tokens_before} tokens → 压缩={compressed} → {ctx.estimated_tokens} tokens")
    
    print()
    print(f"最终统计：")
    stats = ctx.stats()
    for k, v in stats.items():
        print(f"  {k}: {v}")


await simulate_agent_with_compression(
    "分析 src/ 目录下所有 Python 文件的代码质量",
    num_rounds=3,
)

## 10. 小结

| 模块 | 作用 | 关键细节 |
|------|------|----------|
| `format_history_for_compression` | 把消息列表转成 LLM 可读的文本 | tool 结果截断到 500 字符；跳过 system 消息 |
| `ChatCompactor.compress` | 调用 LLM 生成摘要 | 非流式调用；专用 system prompt 强调保留技术细节 |
| `prune_tool_outputs` | 压缩前剪掉旧的大型工具结果 | 保护最近 40k tokens；只删 >= 20k tokens 的工具消息 |
| `compress_if_needed` | 压缩触发与替换的主入口 | 先剪枝再压缩；替换后只留一条摘要消息 |

**设计取舍**：摘要替换会造成信息丢失——某些细节 LLM 可能摘要不进去。实际生产中可以把原始历史存到磁盘，压缩只是 context 里的替代视图。

**下一章**：把前面八章的所有模块组装成一个完整的 Coding Agent，实现任务分解、工具调用循环、错误恢复和最终答案生成。

---
## 附：保存到 src/context_compression.py

In [ ]:
import os

# 确保 src 目录存在
os.makedirs("../src", exist_ok=True)

compression_code = '''
# context_compression.py — 对话历史压缩
# 使用: from src.context_compression import ChatCompactor, compress_if_needed, prune_tool_outputs

from src.llm_client import LLMClient
from src.context_manager import ContextManager, MessageItem, UsageStats, count_tokens


def format_history_for_compression(messages: list) -> str:
    """
    把历史消息列表格式化为可供压缩的纯文本。
    跳过 system 消息，tool 结果截断到 500 字符。
    """
    MAX_TOOL_CHARS = 500
    lines = []

    for msg in messages:
        role = msg.role
        content = msg.content

        # 系统消息不参与压缩
        if role == "system":
            continue

        # 工具结果截断
        if role == "tool":
            if len(content) > MAX_TOOL_CHARS:
                content = content[:MAX_TOOL_CHARS] + f"\\n...[工具结果已截断，原长 {len(msg.content)} 字符]"

        # assistant 有工具调用时，补充工具调用信息
        if role == "assistant" and msg.tool_calls:
            tool_names = [tc.get("function", {}).get("name", "?") for tc in msg.tool_calls]
            tool_info = f" [调用工具: {\', \'.join(tool_names)}]"
            content = (content or "") + tool_info

        lines.append(f"[{role}]: {content}")

    return "\\n\\n".join(lines)


def prune_tool_outputs(
    messages: list,
    protect_last_tokens: int = 40_000,
    min_save_tokens: int = 20_000,
) -> list:
    """
    剪掉历史中较旧的大型工具结果，减少压缩请求的 token 消耗。
    保护最近 protect_last_tokens token 内的消息不动。
    只删除节省 >= min_save_tokens 的工具消息。
    """
    if not messages:
        return messages

    # 从后往前计算，找保护区起点
    cumulative = 0
    protect_from_idx = len(messages)
    for i in range(len(messages) - 1, -1, -1):
        cumulative += messages[i].token_count
        if cumulative >= protect_last_tokens:
            protect_from_idx = i
            break

    result = []
    for i, msg in enumerate(messages):
        if i >= protect_from_idx:
            result.append(msg)
        elif msg.role == "tool" and msg.token_count >= min_save_tokens:
            pass  # 删除旧的大型工具结果
        else:
            result.append(msg)

    return result


class ChatCompactor:
    """
    调用 LLM 把对话历史压缩成摘要。
    保留所有技术细节：文件名、变量名、错误信息、已完成的操作。
    """

    COMPRESSOR_SYSTEM = (
        "你是对话历史压缩器。"
        "把下面的对话历史压缩成简洁的摘要，保留所有重要的技术细节、"
        "文件名、变量名、错误信息、已执行的操作和结果。"
        "摘要要足够详细，让后续的 AI 能继续完成任务。"
        "直接输出摘要内容，不要加任何前缀或说明。"
    )

    def __init__(self, llm: LLMClient):
        self.llm = llm

    async def compress(self, ctx: ContextManager) -> tuple[str, UsageStats]:
        """压缩 ctx 中的历史消息，返回 (摘要字符串, token 使用量)。"""
        history_text = format_history_for_compression(ctx._messages)

        if not history_text.strip():
            return "（无历史记录）", UsageStats()

        compression_messages = [
            {"role": "system", "content": self.COMPRESSOR_SYSTEM},
            {"role": "user", "content": history_text},
        ]

        summary, token_usage = await self.llm.chat_completion(
            compression_messages,
            stream=False,
        )

        usage_stats = UsageStats(
            prompt_tokens=token_usage.prompt_tokens,
            completion_tokens=token_usage.completion_tokens,
            total_tokens=token_usage.total_tokens,
        )

        return summary, usage_stats


async def compress_if_needed(
    ctx: ContextManager,
    compactor: ChatCompactor,
) -> bool:
    """
    检查是否需要压缩，需要则执行。返回 True 表示执行了压缩。
    流程：检查阈值 → 剪枝 → LLM 摘要 → 替换历史。
    """
    if not ctx.needs_compression():
        return False

    # 先剪枝，删旧的大型工具结果
    ctx._messages = prune_tool_outputs(ctx._messages)

    # 调用 LLM 生成摘要
    summary, usage = await compactor.compress(ctx)

    # 替换历史
    ctx.clear_messages()
    ctx.add_assistant_message(f"[对话历史摘要]\\n{summary}")
    ctx.update_usage(usage.prompt_tokens, usage.completion_tokens)

    return True
'''

with open("../src/context_compression.py", "w", encoding="utf-8") as f:
    f.write(compression_code.strip())

print("已保存到 src/context_compression.py")
print()
print("后续使用方式：")
print("  from src.context_compression import ChatCompactor, compress_if_needed, prune_tool_outputs")

await llm.close()